In [2]:
import pandas as pd

# Загрузка данных из CSV файла
df = pd.read_csv('changed_dancer_role_info.csv')

# Инициализация списка для хранения результатов
results = []

# Группировка данных по 'dancer_id' и сортировка по 'update_date'
grouped = df.sort_values(by='update_date').groupby('dancer_id')

# Функция для определения перехода между дивизионами
def determine_transition(prev, curr):
    divisions = ['Novice', 'Intermediate', 'Advanced', 'All-Star', 'Champion']
    if prev in divisions and curr in divisions:
        prev_index = divisions.index(prev)
        curr_index = divisions.index(curr)
        if curr_index > prev_index:
            return prev, curr
    return None, None

# Обработка каждой группы
for dancer_id, group in grouped:
    # Сравнение каждой строки с предыдущей
    for i in range(1, len(group)):
        prev_row = group.iloc[i - 1]
        curr_row = group.iloc[i]
        update_date = curr_row['update_date']
        dancer_name = curr_row['dancer_name']

        # Сравнение полей и определение переходов
        for role_type in ['dominate', 'non_dominate']:
            for aspect in ['required', 'allowed']:
                prev_division = prev_row[f'{role_type}_{aspect}']
                curr_division = curr_row[f'{role_type}_{aspect}']
                prev_div, curr_div = determine_transition(prev_division, curr_division)
                if prev_div and curr_div:
                    transition_type = 'required' if aspect == 'required' else 'allowed'
                    role = curr_row[f'{role_type}_role']
                    results.append({
                        'Дата апдейта': update_date,
                        'Предыдущий дивизион': prev_div,
                        'Текущий дивизион': curr_div,
                        'Характеристика перехода': transition_type,
                        'Роль': role,
                        'ID танцора': dancer_id,
                        'Имя танцора': dancer_name
                    })

# Создание DataFrame из результатов
result_df = pd.DataFrame(results)

# Удаление дубликатов, оставляя только записи с 'required' при наличии дубликатов
result_df = result_df.sort_values(by='Характеристика перехода', ascending=False).drop_duplicates(
    subset=['Дата апдейта', 'Предыдущий дивизион', 'Текущий дивизион', 'Роль', 'ID танцора', 'Имя танцора'],
    keep='first'
)

# Сохранение результатов в новый CSV файл
result_df.to_csv('dancer_transitions.csv', index=False)


AttributeError: 'DataFrame' object has no attribute 'append'